In [ ]:
import numpy as np 
import pandas as pd 
import MLM.angle_between as angle
import MLM.moire_lattice_vector_finder as mlf 
import MLM.structure_writer as sw
from matplotlib.path import Path
from MLM.match import run_and_filter
from ase import Atoms
from ase.io import write
from fractional_boundry_test import select_atoms_fractional

In [2]:
file_1 = "../moire_structures/mos2/mos2_monolayer.vasp"
file_2 = "../moire_structures/mos2/mos2_monolayer.vasp"
file_3 = "../moire_structures/mos2/mos2_monolayer.vasp"

In [3]:
lattice_vectors1, atom_type_arr1, dat1 = mlf.read_vasp(file_1)

lattice_vectors2, atom_type_arr2, dat2 = mlf.read_vasp(file_2)

lattice_vectors3, atom_type_arr3, dat3 = mlf.read_vasp(file_3)


In [4]:
candidate = pd.read_pickle('../moire_structures/mos2/mos2_ttl.pkl')

candidate

,Theta,Phi,A1,A2,delvec,delang,norm_A1,norm_A2
10,21.79,38.21,"[11.080997057200001, 19.192852852599998]","[-11.081002168600001, 19.192852852599998]",0.001244,3.814877e-06,22.161997,22.161999
11,21.79,43.57,"[3.1659969686, 21.9346889744]","[20.5789981858, 8.2255083654]",0.001387,7.318076e-06,22.161997,22.161998
17,21.79,49.58,"[17.4129961058, 24.6765250962]","[-12.6640032086, 27.418361217999998]",0.001352,4.946715e-06,30.201711,30.201714
18,27.80,49.58,"[17.4129961058, 24.6765250962]","[30.076999314400005, -2.7418361218000005]",0.001352,2.557022e-06,30.201711,30.201714
19,21.79,53.99,"[1.5829959286, 30.1601973398]","[26.9109972344, 13.709180609]",0.000540,7.587588e-06,30.201712,30.201712
20,32.20,53.99,"[-1.5829959286000008, -30.1601973398]","[25.3280013058, -16.4510167308]",0.000540,3.101992e-06,30.201712,30.201715
37,21.79,34.96,"[7.9149949772, 35.6438695834]","[34.825997323, 10.967344487200002]",0.000220,6.912406e-06,36.512088,36.512090
38,13.17,34.96,"[7.9149949772, 35.643869583400004]","[-26.9110023458, 24.676525096200002]",0.000220,6.595630e-07,36.512088,36.512093
43,27.80,55.59,"[1.582994468199999, 41.127541827]","[36.4089961724, 19.192852852599998]",0.001115,7.606940e-06,41.157995,41.157996
53,38.21,56.11,"[1.5829937379999999, 46.6112140706]","[41.1579956414, 21.9346889744]",0.000188,7.611934e-06,46.638087,46.638087


In [5]:
df = candidate

In [6]:
import warnings
warnings.filterwarnings('ignore')


count = 1
for row in df.iterrows():
    A1 = np.array(row[1]['A1'])
    A2 = np.array(row[1]['A2'])
    delvec = row[1]['delvec']
    print("Delvec is ", delvec)
    norm = row[1]['norm_A1'] + row[1]['norm_A2']
    replicate = int((norm/(min(np.linalg.norm(lattice_vectors1['a']),np.linalg.norm(lattice_vectors2['a'])))))*8

    new_lattice_vectors1, new_total_atoms1, replicated_atom1, replicated_type_arr1 = sw.replicate_atoms(a = lattice_vectors1['a'],
                                                                                             b = lattice_vectors1['b'],
                                                                                             c = lattice_vectors1['c'],
                                                                                             atom_data = dat1,
                                                                                             atom_type_arr = atom_type_arr1,
                                                                                             natoms = len(dat1),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)
    new_lattice_vectors2, new_total_atoms2, replicated_atom2, replicated_type_arr2 = sw.replicate_atoms(a = lattice_vectors2['a'],
                                                                                             b = lattice_vectors2['b'],
                                                                                             c = lattice_vectors2['c'],
                                                                                             atom_data = dat2,
                                                                                             atom_type_arr = atom_type_arr2,
                                                                                             natoms = len(dat2),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)
    new_lattice_vectors3, new_total_atoms3, replicated_atom3, replicated_type_arr3 = sw.replicate_atoms(a = lattice_vectors3['a'],
                                                                                             b = lattice_vectors3['b'],
                                                                                             c = lattice_vectors3['c'],
                                                                                             atom_data = dat3,
                                                                                             atom_type_arr = atom_type_arr3,
                                                                                             natoms = len(dat3),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)

    # Create a DataFrame for positions
    positions1_df = pd.DataFrame(replicated_atom1, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions1_df['type'] = replicated_type_arr1
    
    positions2_df = pd.DataFrame(replicated_atom2, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions2_df['type'] = replicated_type_arr2 
    positions2_df['z'] = positions2_df['z'] + 3 + (positions1_df['z'].max() )
    
    positions3_df = pd.DataFrame(replicated_atom3, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions3_df['type'] = replicated_type_arr3 
    positions3_df['z'] = positions3_df['z'] + 3 + (positions2_df['z'].max() )
    

    # Rotating the second or middle layer
    theta1 = row[1]['Theta']
    
    theta1_rad = np.deg2rad(theta1)
    
    rotation_matrix_1 = np.array([[np.cos(theta1_rad), -np.sin(theta1_rad), 0],
                            [np.sin(theta1_rad), np.cos(theta1_rad), 0],
                            [0, 0, 1]])
    
    middle_layer_positions = positions2_df[['x', 'y', 'z']].values  # Extract x, y, z positions
    middle_rotated_positions = middle_layer_positions @ rotation_matrix_1.T  # Apply rotation

    # Create a new DataFrame for the rotated positions
    rotated_middle_layer = pd.DataFrame(middle_rotated_positions, columns=['x', 'y', 'z'])

    # Add the atom types back to the rotated DataFrame
    rotated_middle_layer['type'] = positions2_df['type'].values
    
    # Rotating the top or third layer
    theta2 = row[1]['Phi']
    
    theta2_rad = np.deg2rad(theta2)
    
    rotation_matrix_2 = np.array([[np.cos(theta2_rad), -np.sin(theta2_rad), 0],
                            [np.sin(theta2_rad), np.cos(theta2_rad), 0],
                            [0, 0, 1]])
    
    top_layer_positions = positions3_df[['x', 'y', 'z']].values  # Extract x, y, z positions
    top_rotated_positions = top_layer_positions @ rotation_matrix_2.T  # Apply rotation

    # Create a new DataFrame for the rotated positions
    rotated_top_layer = pd.DataFrame(top_rotated_positions, columns=['x', 'y', 'z'])

    # Add the atom types back to the rotated DataFrame
    rotated_top_layer['type'] = positions3_df['type'].values
    
    
    
    # Concatenate DataFrames vertically
    concat_vertical = pd.concat([positions1_df, rotated_middle_layer, rotated_top_layer], ignore_index=True)
    
    z_dim = (positions3_df['z'].max() - positions1_df['z'].min()) + 50
    

    A3 = np.array([0, 0, z_dim])
    
    vertex = [ [0.0,0.0], [A1[0], A1[1]],  [(A1+A2)[0],(A1+A2)[1]], [A2[0],A2[1]]]
    
    polygon_points = vertex
    
    polygon_path = Path(polygon_points)
    
    # selected_atoms_df = sw.select_atoms_in_polygon_path(concat_vertical, polygon_path, tolerance=-0.0)
    
    selected_atoms_df = select_atoms_fractional(concat_vertical, A1, A2, dtol=delvec)
    
    # unique_types = selected_atoms_df['type'].unique()
    
    # sort_order = {value: index for index, value in enumerate(unique_types)}
    
    # selected_atoms_df['sort_key'] = selected_atoms_df['type'].map(sort_order)
    
    # sorted_df = selected_atoms_df.sort_values(by='sort_key').drop(columns='sort_key').copy()
    
    # sorted_df = sorted_df.round(5)
    
    # final_df = selected_atoms_df.round(5).copy()
    
    sw.write_lammps_ase(f"../moire_structures/mos2/TTL_mos2_{theta1}_{theta2}.dat",A1,A2,A3, selected_atoms_df)
    print(f"Done with {count} | {np.round(theta1,2)} | {np.round(theta2,2)}")
    count = count + 1
    
    
    
    


Delvec is  0.001243555749335279
Done with 1 | 21.79 | 38.21
Delvec is  0.001386549318922247
Done with 2 | 21.79 | 43.57
Delvec is  0.0013521424329268647
Done with 3 | 21.79 | 49.58
Delvec is  0.0013521424334265984
Done with 4 | 27.8 | 49.58
Delvec is  0.0005397325890714131
Done with 5 | 21.79 | 53.99
Delvec is  0.000539732593826531
Done with 6 | 32.2 | 53.99
Delvec is  0.00022014244856911365
Done with 7 | 21.79 | 34.96
Delvec is  0.00022014245006088233
Done with 8 | 13.17 | 34.96
Delvec is  0.0011150546721471846
Done with 9 | 27.8 | 55.59
Delvec is  0.00018791398304131198
Done with 10 | 38.21 | 56.11
Delvec is  0.00018791399051837758
Done with 11 | 17.9 | 56.11
Delvec is  0.0005849138454646814
Done with 12 | 27.8 | 40.97
Delvec is  0.0005849138433149471
Done with 13 | 13.17 | 40.97


# LAMMPS Writer

In [59]:
selected_atoms_df.head()

,x,y,z,type,sort_key,numeric_id,mass
1218,-0.237450,1.416615,12.273334,Mo,0,2,95.940
1222,-0.237450,3.244506,10.696824,S,1,1,32.065
1223,-0.237450,3.244506,13.850458,S,1,3,32.065
1305,1.345549,4.158452,12.273334,Mo,0,2,95.940
1306,2.928549,3.244506,10.696824,S,1,1,32.065


In [57]:
len(selected_atoms_df)

42

## Set layer id as required by KC forcefield

In [58]:
unique_z = sorted(selected_atoms_df['z'].unique())

numeric_id_dict = {value: index+1 for index, value in enumerate(unique_z)}

print(f"Numeric Id: {numeric_id_dict}")

# Create a new column to map the sort order
selected_atoms_df['numeric_id'] = selected_atoms_df['z'].map(numeric_id_dict)




Numeric Id: {10.696823867: 1, 12.273333597: 2, 13.850457828: 3, 16.696823867: 4, 18.273333597: 5, 19.850457828: 6}


/tmp/ipykernel_784534/4183766918.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_atoms_df['numeric_id'] = selected_atoms_df['z'].map(numeric_id_dict)


## Set Masses Requires manual input

In [60]:
unique_element = selected_atoms_df['type'].unique()

print(f"Unique Elements: {unique_element}")



Unique Elements: ['Mo' 'S']


In [61]:
mass_type_dict = {'B': 10.811, 'C': 12.011, 'N': 14.007, 'O': 15.999, 
                  'S': 32.065, 'Mo': 95.94, 'W': 183.84, 'Se': 78.96, 
                  'Te': 127.6, 'Pb': 207.2, 'Bi': 208.980, 'Ti': 47.867}

In [62]:

selected_atoms_df['mass'] = selected_atoms_df['type'].map(mass_type_dict)

/tmp/ipykernel_784534/3153881743.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  selected_atoms_df['mass'] = selected_atoms_df['type'].map(mass_type_dict)


In [63]:
selected_atoms_df.head()

,x,y,z,type,sort_key,numeric_id,mass
1218,-0.237450,1.416615,12.273334,Mo,0,2,95.940
1222,-0.237450,3.244506,10.696824,S,1,1,32.065
1223,-0.237450,3.244506,13.850458,S,1,3,32.065
1305,1.345549,4.158452,12.273334,Mo,0,2,95.940
1306,2.928549,3.244506,10.696824,S,1,1,32.065


In [64]:
sorted(selected_atoms_df['numeric_id'].unique())

[1, 2, 3, 4, 5, 6]

In [65]:
def write_lammps_data(filename, A1, A2, A3, atom_data, mass_type_dict):
    
    angle_A1 = angle.angle_between(A1,np.array([1,0,0]))
    angle_A2 = angle.angle_between(A2,np.array([1,0,0]))
    if angle_A1 < angle_A2:
        rot_rad = np.deg2rad(angle_A1)
    else:
        rot_rad = np.deg2rad(angle_A2)
    
    rotation_matrix = np.array([[np.cos(-rot_rad), -np.sin(-rot_rad), 0],
                            [np.sin(-rot_rad), np.cos(-rot_rad), 0],
                            [0, 0, 1]])

    A11 = np.round((rotation_matrix @ A1.T),4)
    A22 = np.round((rotation_matrix @ A2.T),4)
    
    type_counts = sorted(atom_data['numeric_id'].unique())
    lines = [
        "System Generated by MLM \n\n",
        f"{len(atom_data)} atoms\n",
        f"{int(atom_data['numeric_id'].nunique())} atom types\n\n",
        f"0.0 {max(A11[0],A22[0],A3[0])} xlo xhi\n"
        f"0.0 {max(A11[1],A22[1],A3[1])} ylo yhi\n",
        f"0.0 {max(A11[2],A22[2],A3[2])} zlo zhi\n",
        f"{sorted([A11[0],A22[0],A3[0]],reverse=True)[1]} 0.0 0.0 xy xz yz\n\n",
        "Masses\n\n",
    ]
    # Loop to print each element type and count 
    for key in type_counts:
        mass_value = atom_data.loc[atom_data['numeric_id'] == key, 'mass'].values[0] 
        lines.append(f"{key} {mass_value}\n") 
        
    lines.extend([ "\nAtoms #atomic \n\n" ])
    
    atomic_positions = atom_data[['x', 'y', 'z']].values
    rotated_atomic_pos = atomic_positions @ rotation_matrix.T
    rot_atomic_positions = pd.DataFrame(rotated_atomic_pos, columns=['x', 'y', 'z'])
    rot_atomic_positions.reset_index(inplace=True)
    rot_atomic_positions['id'] = rot_atomic_positions.index + 1
    rot_atomic_positions['numeric_id'] = atom_data['numeric_id'].values
    data = rot_atomic_positions[['id', 'numeric_id', 'x', 'y', 'z']]
    
    
    # Write the buffer to the file
    with open(filename, 'w') as file:
        file.writelines(lines)
        # Use to_csv for efficient DataFrame writing
        data.to_csv(file, sep=' ', index=False, header=False, mode='a')

In [66]:
write_lammps_data("lammps_test_mos2_21.8.data", A1, A2, A3, selected_atoms_df, mass_type_dict)

In [31]:
np.linalg.norm(np.array([-8.6355194156, -2.1367369111]))

np.float64(8.895945155207608)

In [32]:
np.linalg.norm(np.array([-2.4672912616, -8.5469476444]))

np.float64(8.895945155276218)

In [33]:
mlf.angle_between(np.array([-8.6355194156, -2.1367369111]),np.array([-2.4672912616, -8.5469476444]) )

60.000000000017

In [20]:
filtered_df.head(n=10)

,Theta,Phi,A1,A2,delvec,delang,norm
0,6.02,12.68,"[54.77781833999998, -332.57961135000005]","[332.57961135000005, 54.77781833999998]",0.002256,0.0,476.675586
18,6.02,14.25,"[-242.58748122000003, -348.23041659]","[348.23041659, -242.58748122000003]",0.000242,0.0,600.188486
32,6.02,16.26,"[50.86511702999999, -258.23828646000004]","[258.23828646000004, 50.86511702999999]",0.000940,0.0,372.221098
74,6.02,22.62,"[46.95241571999999, -183.89696157]","[183.89696157, 46.95241571999999]",0.000447,0.0,268.412451
168,6.02,24.95,"[269.97639039, -172.15885764]","[-172.15885764, -269.97639039]",0.001226,0.0,452.826509
186,6.02,30.51,"[403.00823493, 133.03184453999998]","[133.03184453999998, -403.00823493]",0.001758,0.0,600.188486
198,6.02,36.87,"[109.55563668, 43.039714409999995]","[43.039714409999995, -109.55563668]",0.000210,0.0,166.462335
310,6.02,53.13,"[-31.301610480000008, -113.46833799000001]","[113.46833799000001, -31.301610480000008]",0.000210,0.0,166.462335
416,6.02,59.49,"[414.74633886000004, -89.99213013000002]","[-89.99213013000002, -414.74633886000004]",0.001758,0.0,600.188486
430,6.02,67.38,"[-27.388909170000005, -187.80966288000002]","[187.80966288000002, -27.388909170000005]",0.000447,0.0,268.412451


In [44]:
import warnings
warnings.filterwarnings('ignore')

df = final_df.head(n=1)

count = 1
for row in df.iterrows():
    A1 = np.array(row[1]['A1'])
    A2 = np.array(row[1]['A2'])
    norm = row[1]['norm'] + row[1]['norm']
    replicate = int((4*norm/np.linalg.norm(lattice_vectors1['a'])))
    new_lattice_vectors1, new_total_atoms1, replicated_atom1, replicated_type_arr1 = sw.replicate_atoms(a = lattice_vectors1['a'],
                                                                                             b = lattice_vectors1['b'],
                                                                                             c = lattice_vectors1['c'],
                                                                                             atom_data = dat1,
                                                                                             atom_type_arr = atom_type_arr1,
                                                                                             natoms = len(dat1),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)
    new_lattice_vectors2, new_total_atoms2, replicated_atom2, replicated_type_arr2 = sw.replicate_atoms(a = lattice_vectors2['a'],
                                                                                             b = lattice_vectors2['b'],
                                                                                             c = lattice_vectors2['c'],
                                                                                             atom_data = dat2,
                                                                                             atom_type_arr = atom_type_arr2,
                                                                                             natoms = len(dat2),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)
    new_lattice_vectors3, new_total_atoms3, replicated_atom3, replicated_type_arr3 = sw.replicate_atoms(a = lattice_vectors3['a'],
                                                                                             b = lattice_vectors3['b'],
                                                                                             c = lattice_vectors3['c'],
                                                                                             atom_data = dat3,
                                                                                             atom_type_arr = atom_type_arr3,
                                                                                             natoms = len(dat3),
                                                                                             na = replicate,
                                                                                             nb = replicate,
                                                                                             nc = 1)

    # Create a DataFrame for positions
    positions1_df = pd.DataFrame(replicated_atom1, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions1_df['type'] = replicated_type_arr1
    
    positions2_df = pd.DataFrame(replicated_atom2, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions2_df['type'] = replicated_type_arr2 
    positions2_df['z'] = positions2_df['z'] + 3 + (positions1_df['z'].max()  )
    
    # Create a DataFrame for positions
    positions3_df = pd.DataFrame(replicated_atom3, columns=['x', 'y', 'z'])

    # Add the atom types to the DataFrame
    positions3_df['type'] = replicated_type_arr3
    positions3_df['z'] = positions3_df['z'] + 3 + (positions2_df['z'].max() )
    

    
    theta1 = row[1]['Theta']

    print(f"Twist Angle: {theta1}")

    theta_rad = np.deg2rad(theta1)

    print(f"Twist Angle in Radians: {theta_rad}")

    rotation_matrix = np.array([[np.cos(theta_rad), -np.sin(theta_rad), 0],
                                [np.sin(theta_rad), np.cos(theta_rad), 0],
                                [0, 0, 1]])



    # Rotate the positions in the upper layer
    middle_layer_positions = positions2_df[['x', 'y', 'z']].values  # Extract x, y, z positions
    rotated_positions = middle_layer_positions @ rotation_matrix.T  # Apply rotation

    # Create a new DataFrame for the rotated positions
    rotated_middle_layer = pd.DataFrame(rotated_positions, columns=['x', 'y', 'z'])

    # Add the atom types back to the rotated DataFrame
    rotated_middle_layer['type'] = positions2_df['type'].values
    
    theta2 =   row[1]['Phi']
    print(f"Twist Angle: {theta2}")

    theta_rad = np.deg2rad(theta2)

    rotation_matrix = np.array([[np.cos(theta_rad), -np.sin(theta_rad), 0],
                                [np.sin(theta_rad), np.cos(theta_rad), 0],
                                [0, 0, 1]])



    # Rotate the positions in the upper layer
    upper_layer_positions = positions3_df[['x', 'y', 'z']].values  # Extract x, y, z positions
    rotated_positions = upper_layer_positions @ rotation_matrix.T  # Apply rotation

    # Create a new DataFrame for the rotated positions
    rotated_upper_layer = pd.DataFrame(rotated_positions, columns=['x', 'y', 'z'])

    # Add the atom types back to the rotated DataFrame
    rotated_upper_layer['type'] = positions3_df['type'].values
    
    
    
    # Concatenate DataFrames vertically
    concat_vertical = pd.concat([positions1_df, rotated_middle_layer, rotated_upper_layer], ignore_index=True)
    
    z_dim = (positions2_df['z'].max() - positions1_df['z'].min()) + 50
    

    A3 = np.array([0, 0, z_dim])
    
    vertex = [ [0.0,0.0], [A1[0], A1[1]],  [(A1+A2)[0],(A1+A2)[1]], [A2[0],A2[1]]]
    
    polygon_points = vertex
    
    polygon_path = Path(polygon_points)
    
    selected_atoms_df = sw.select_atoms_in_polygon_path(concat_vertical, polygon_path, tolerance=-0.02)
    
    # unique_types = selected_atoms_df['type'].unique()
    
    # sort_order = {value: index for index, value in enumerate(unique_types)}
    
    # selected_atoms_df['sort_key'] = selected_atoms_df['type'].map(sort_order)
    
    # sorted_df = selected_atoms_df.sort_values(by='sort_key').drop(columns='sort_key').copy()
    
    # sorted_df = sorted_df.round(5)
    
    final_df = selected_atoms_df.round(5).copy()
    
    sw.write_lammps_ase(f"../moire_structures/STO/STO_tri_layer/sto_helical_36.86_{count}.dat",A1,A2,A3, final_df)
    print(f"Done with {count} | {np.round(theta1,2)}")
    count = count + 1
    # break
    
    
    


Twist Angle: 36.87
Twist Angle in Radians: 0.6435028952103092
Twist Angle: 73.72600000000122
Done with 1 | 36.87
